# Data profiling deep dive

`freshdata` profiling surfaces missingness, dtype issues, and duplicates,
and can attach a faithful dry-run cleaning plan. Because profiling runs the
same inference as `clean`, the preview matches the result.

In [ ]:
import numpy as np
import pandas as pd
import freshdata as fd

rng = np.random.default_rng(0)
n = 300
df = pd.DataFrame({
    'customer_id': range(n),
    'age': rng.normal(40, 12, n),
    'income': rng.normal(60000, 15000, n),
    'segment': rng.choice(['A', 'B', 'C'], n),
    'signup': pd.date_range('2022-01-01', periods=n, freq='D').astype(str),
})
df.loc[rng.choice(n, 60, replace=False), 'income'] = np.nan
df.loc[rng.choice(n, 30, replace=False), 'segment'] = None
df.head()

## Human-readable profile

In [ ]:
print(fd.profile(df))

## Programmatic access to column-level quality

In [ ]:
profile = fd.profile(df)
for col in profile.columns:
    print(f'{col.name:14s} dtype={col.dtype!s:10s} missing={col.missing_pct:.0f}%')

## Inferred roles and a dry-run plan

In [ ]:
print(fd.infer_roles(df))
print()
print(fd.suggest_plan(df, id_columns=('customer_id',)).summary())

## Compare strategies before committing

In [ ]:
print(fd.compare_plans(df))